In [ ]:
from utils.postgres_util import get_engine_from_env
from utils.producer_queue_manager import (
    ensure_send_queue_runtime_columns,
    ensure_simulation_state_control_table,
    upsert_simulation_state_control,
    read_simulation_state_control,
    claim_pending_send_queue_batch,
    mark_claimed_batch_sent, 
    mark_claimed_batch_failed,
    requeue_failed_messages,
    release_stale_claims,
    get_send_queue_status_counts
)



In [ ]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

IS_ENABLED_FLAG = True

PRODUCER_TOPIC = str("pump.telemetry.synthetic")
PRODUCER_BATCH_SIZE = int(5200)
PRODUCER_POLL_SECONDS = float(0.0)
PRODUCER_MAX_SEND_ATTEMPTS = int(3)

PRODUCER_WORKER_ID = str("producer_worker_001")

SIMULATION_TABLE_NAME = str("simulation_state_control")
SEND_QUEUE_TABLE_NAME = str("synthetic_sensor_messages_send_queue")

CLAIM_BATCH_SIZE = int(500)
STALE_CLAIM_RELEASE_MINUTES = int(15)

In [ ]:

engine = get_engine_from_env()


In [ ]:

ensure_send_queue_runtime_columns(
    engine=engine,
    schema = SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)


In [ ]:

ensure_simulation_state_control_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)


In [ ]:

upsert_simulation_state_control(
    engine=engine,
    dataset_id = DATASET_ID,
    run_id = RUN_ID,
    is_enabled = IS_ENABLED_FLAG,
    producer_topic = PRODUCER_TOPIC,
    producer_batch_size = PRODUCER_BATCH_SIZE,
    producer_poll_seconds = PRODUCER_POLL_SECONDS,
    max_send_attempts = PRODUCER_MAX_SEND_ATTEMPTS,
    schema = SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)


In [ ]:

control_row = read_simulation_state_control(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)


In [ ]:

control_row

In [ ]:
claimed_dataframe = claim_pending_send_queue_batch(
    engine=engine,
    batch_size=CLAIM_BATCH_SIZE,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
    producer_topic=PRODUCER_TOPIC,
    producer_worker_id=PRODUCER_WORKER_ID,
)

print("Claimed rows:", len(claimed_dataframe))



In [ ]:
display(
    claimed_dataframe[
        [
            "claim_token",
            "observation_index",
            "message_sequence_index",
            "sensor_name",
            "sensor_index",
            "queue_status",
            "producer_topic",
            "producer_worker_id",
        ]
    ].head(20)
)

In [ ]:
if not claimed_dataframe.empty:
    claim_token = str(claimed_dataframe["claim_token"].iloc[0])

    sent_dataframe = mark_claimed_batch_sent(
        engine=engine,
        claim_token=claim_token,
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Marked sent:", len(sent_dataframe))
    display(sent_dataframe)
else:
    print("No claimed rows to mark as sent.")

In [ ]:
if not claimed_dataframe.empty:
    claim_token = str(claimed_dataframe["claim_token"].iloc[0])

    failed_dataframe = mark_claimed_batch_failed(
        engine=engine,
        claim_token=claim_token,
        error_message="Kafka publish failed during test run.",
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Marked failed:", len(failed_dataframe))
    display(failed_dataframe.head())
else:
    print("No claimed rows to mark as failed.")

In [ ]:
requeued_dataframe = requeue_failed_messages(
    engine=engine,
    max_send_attempts = PRODUCER_MAX_SEND_ATTEMPTS,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

print("Requeued failed rows:", len(requeued_dataframe))
display(requeued_dataframe.head())

In [ ]:
released_dataframe = release_stale_claims(
    engine=engine,
    stale_after_minutes=STALE_CLAIM_RELEASE_MINUTES,
    max_send_attempts=PRODUCER_MAX_SEND_ATTEMPTS,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

print("Released stale claims:", len(released_dataframe))
display(released_dataframe.head())

In [ ]:
status_dataframe = get_send_queue_status_counts(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

display(status_dataframe)